# XI.2 - Travaux pratiques

L'objectif de cette séance de travaux pratiques est d'étendre le modèle de régression logistique du TP précédent à un réseau de neurones plus complexe. Précédemment, nous avons implémenté à la main une régression logistique, c'est-à-dire le plus simple des réseaux de neurones linéaires. Pour cette séance, nous allons implémenter un perceptron multi-couches (*multi-layer perceptron* ou MLP). Contrairement à la régression logistique, qui se limite à des séparateurs linéaires, le perceptron permet d'apprendre des frontières de décisions non-linéaires. En outre, la famille des perceptrons est un approximateur universels des fonctions continues. Autrement dit, il s'agit d'un modèle très expressif, à la base de l'apprentissage profond.

Nous allons voir dans un premier temps comment implémenter la phase *forward* (passe avant) pour produire des prédictions, puis la phase *forward* (passe arrière) pour l'optimisation d'un perceptron à une seule couche cachée.

Commençons par charger à nouveau le jeu de données MNIST :

In [9]:
from tensorflow.keras.datasets import mnist
import numpy as np

# Charger MNIST
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Créer une permutation aléatoire
indices = np.random.permutation(len(x_train))
print(indices.shape)
# Mélanger les images et les labels de la même manière
x_train = x_train[indices]
y_train = y_train[indices]

(60000,)


In [10]:
from keras.datasets import mnist
# Import de MNIST depuis Keras
(X_train, y_train), (X_test, y_test) = mnist.load_data()
# Transformation des images 28x28 en vecteur de dimension 784
X_train = X_train.reshape(60000, 784).astype('float32')
X_test = X_test.reshape(10000, 784).astype('float32')
# Normalisation entre 0 et 1
X_train /= 255
X_test /= 255


## Prédiction avec un perceptron (*forward*)

L'architecture du perceptron à une seule couche cachée est illustrée dans la figure ci-dessous.

Comme la dernière fois, on considère les données de la base MNIST. Chaque image est représentée par un vecteur de taille $28^2=784$. Le perceptron applique différentes opérations mathématiques pour transformer l'entrée et produire la prédiction finale, c'est-à-dire la catégorie de l'image :

1. Une projection linéaire, qui va projeter chaque image sur un vecteur de dimensions $(1, L)$. $L$ représente ici la largeur (le nombre de neurones) de la couche cachée du perceptron, par exemple $L=100$. En considérant que chaque exemple $\mathbf{x_i}$ est un vecteur ligne $(1,784)$, la projection linéaire est représentée par une matrice $\mathbf{W^h}$ $(784, L)$ et un vecteur de biais $\mathbf{b^h}$ $(1, L)$. La projection s'écrit:

   $$
   \mathbf{u_i} = \mathbf{x_i} \mathbf{W^h} + \mathbf{b^h}.
   $$

2. L'application d'une fonction de transfert non-linéaire, par exemple une sigmoïde :

   $$
   \forall j \in \left\lbrace 1; L \right\rbrace, ~ h_{i,j} = \frac{1}{1+\exp(-u_{i,j})}.
   $$

3. Une deuxième projection linéaire, qui va projeter la représentation interne (les activations de la couche cachée) de dimensions $(1,L)$ en un vecteur de $(1, K)$, avec $K$ le nombre de classes considérées (ici, 10). $K$ représente le nombre de neurones en sortie, c'est-à-dire la dimensionalité du vecteur prédit. Cette opération de projection linéaire est représentée par la matrice $\mathbf{W^y}$ de dimensions $(L, K)$ et le vecteur de biais $\mathbf{b^y}$ de dimensions $(1, K)$). Matriciellement, la projection est représentée par l'opération :

   $$
   \mathbf{v_i} =\mathbf{h_i} \mathbf{W^y} + \mathbf{b^y}.
   $$

4. Enfin, l'application d'une non-linéarité *softmax*. Comme pour la régression logistique, cela permet de transformer les activations de sortie en probabilités pour une distribution catégorielle :

   $$
   \forall j \in \left\lbrace 1; K \right\rbrace ~ y_{i,j} = \frac{\exp(v_{i,j})}{\sum\limits_{k=1}^K \exp(v_{i,k})}.
   $$

Notre objectif pour cette séance va être d'implémenter un perceptron (et son apprentissage) sur la base MNIST. Commençons par transformer les étiquettes en vecteur encodé au format *one-hot*:

In [11]:
from keras.utils import to_categorical
import numpy as np
n_classes = 10
L=100 
# Conversion des étiquettes au format one-hot
Y_train = to_categorical(y_train, n_classes)
Y_test = to_categorical(y_test, n_classes)


N, d = X_train.shape # N exemples, dimension d
Wh = np.zeros((d, L)) # initialisation de poids
bh = np.zeros((1, L)) # initialisation des biais
Wy = np.zeros((L, n_classes)) # initialisation de poids
by = np.zeros((1, n_classes)) # initialisation des biais

<div class="admonition-question admonition">

**Question**

En reprenant le squelette du code de la régression logistique, compléter le code ci-dessous pour implémenter la *forward pass* (phase de prédiction) du perceptron multi-couche.

Vous devrez notamment écrire une fonction `forward(batch, Wh, bh, Wy, by)` qui renvoie la prédiction $\hat{\mathbf{y}}$ ainsi que la matrice des activations de la couche cachée. Si l'on considère un batch des données de taille $n \times 784$, les paramètres $\mathbf{W^h}$ ($784\times L$), $\mathbf{b^h}$ ($1\times L$), $\mathbf{W^y}$ ($L\times K$) et $\mathbf{b^y}$ ($1\times K$), la fonction `forward` renvoie :

- la prédiction $\mathbf{\hat{Y}}$ sur le batch ($n\times K$),
- la matrice $\mathbf{H}$ des activations de la couche cachée ($n\times L$),

pour un batch de $n$ exemples.

</div>

In [12]:
def sigmoide(X):
    # Entrée: matrice X de dimensions batch x d
    # Sortie: matrice de mêmes dimensions
    return (1 / (1+np.exp(-X)))
    
def softmax(X):
    E = np.exp(X - np.max(X, axis=1, keepdims=True))
    return E / np.sum(E, axis=1, keepdims=True)


    
def forward(batch, Wh, bh, Wy, by):
    """ Entrées:
        - batch: un batch de n images de MNIST au format vecteur (n, 784)
        - Wh: une matrice de poids entrée -> couche cachée
        - bh: un vecteur de biais pour la couche cachée
        - Wy: une matrice de poids couche cachée -> sortie
        - by: un vecteur de biais pour la sortie
    """

    # H = (Wh * X + bh)
    # Y = (Wy * H + by)
    H = np.matmul(batch, Wh) + bh.repeat(batch.shape[0], axis=0)
    H2 = sigmoide(H)
    
    Y = np.matmul(H2, Wy) + by.repeat(batch.shape[0], axis=0)
    # Calcul du softmax: Y = e^Y / sum(e^Y)
    
    """
    Renvoie:
        - Y_pred: prédictions de sortie
        - H: activations de la couche cachée
    """

    return softmax(Y), H2

print(forward(X_train, Wh, bh,Wy, by)[0].shape)
print(forward(X_train, Wh, bh,Wy, by)[0][1])
    

(60000, 10)
[0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]


## Apprentissage du perceptron (*backward*)

Comme pour la régression logistique, nous allons entraîner le perceptron à l'aide de l'algorithme de descente de gradient. Pour calculer les gradients par rapport aux paramètres de la couche cachée, nous allons avoir besoin d'utiliser l'algorithme de rétro-propagation du gradient (*backpropagation*). Rappellons que pour chaque batch d'exemples, l'algorithme effectue une passe `forward` (implémentée ci-dessus) qui permet de calculer la prédiction du perceptron pour les exemples du batch.

La fonction de coût considérée sera encore l'entropie croisée entre la sortie prédite et les étiquettes de supervision. On calculera alors le gradient de l'erreur par rapport à tous les paramètres du modèle, c'est-à-dire:

- $\mathbf{W^y}$ (dimensions $(L, K)$),
- $\mathbf{b^y}$ (dimensions $(1, K)$),
- $\mathbf{W^h}$ (dimensions $(784, L)$),
- $\mathbf{b^h}$ (dimensions $(1, L)$).

On rappelle ci-dessous les équations des gradients, effectuées depuis la sortie vers l'entrée du réseau :

1. Mise à jour de $\mathbf{W^y}$ et $\mathbf{b^y}$:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{v_i}} = \mathbf{\delta^y_i} = \mathbf{\hat{y_i}} - \mathbf{y_i^*}
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{W^y}} = \frac{1}{n} \mathbf{H}^T (\mathbf{\hat{Y}} - \mathbf{Y^*}) = \frac{1}{n} \mathbf{H}^T \mathbf{\Delta^y} \tag{1}
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b^y}} = \frac{1}{n}\sum_{i=1}^{n}(\mathbf{\hat{y_i}} - \mathbf{y_i^*}) \tag{2}
$$

où $\mathbf{H}$ est la matrice des couches cachées sur le batch ($784 \times L$), $\mathbf{\hat{Y}}$ est la matrice des prédictions sur le batch (taille $n \times K$), $\mathbf{Y^*}$ est la matrice des étiquettes issues de la supervision (*ground truth*, $n \times K$) et $\mathbf{\Delta^y}=\mathbf{\hat{Y}}-\mathbf{Y^*}$.

2. Mise à jour de $\mathbf{W^h}$ et $\mathbf{b^h}$:

<div class="admonition-question admonition">

**Question**

Montrer que

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{u_i}} = \mathbf{\delta^h_i} = \mathbf{\delta^y_i} \mathbf{W^{y}}^T   \odot \sigma^{'}(\mathbf{u_i}) = \mathbf{\delta^y_i} \mathbf{W^{y}}^T \odot (\mathbf{h_i} \odot (1-\mathbf{h_i}))
$$

</div>

<div class="admonition-question admonition">

**Question**

En déduire que les gradients de $\mathcal{L}$ par rapport à $\mathbf{W^h}$ et $\mathbf{b^h}$ s'écrivent matriciellement :

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{W^h}} =  \frac{1}{n} \mathbf{X}^T \mathbf{\Delta^h}
~~~\text{et}~~~
\frac{\partial \mathcal{L}}{\partial \mathbf{b^h}} = \frac{1}{n}\sum_{i=1}^{n}(\delta^h_i)
$$

où $\mathbf{X}$ est la matrice des données sur le batch ($n \times 784$) et $\mathbf{\Delta^h}$ est la matrice des $\delta^h_i$ sur le batch ($n \times L$).

</div>

<div class="admonition-question admonition">

**Question**

Compléter la fonction `backward` ci-dessous qui calcule et renvoie les gradients de l'erreur par rapport aux paramètres du perceptron.

</div>

In [13]:
def cout(Y, A2):
    eps = 1e-15
    A2 = np.clip(A2, eps, 1 - eps)
    return -np.mean(np.sum(Y * np.log(A2), axis=1))
                   
def backward(Y_pred, Y, X, H, Wy):
    """ Entrées:
        - Y_pred: batch de vecteur des prédictions (one-hot)
        - Y: batch de vecteur des étiquettes (one-hot)
        - X: batch d'images (au format vectoriel (n, 784))
        - H: matrice des activations cachées
        - Wy: matrice des poids

        Renvoie:
        - gradWy: gradient de l'erreur (entropie croisée) par rapport à Wy
        - gradby: gradient de l'erreur (entropie croisée) par rapport à by
        - gradWh: gradient de l'erreur (entropie croisée) par rapport à Wh
        - gradbh: gradient de l'erreur (entropie croisée) par rapport à bh
    """
   #Calcul du cout avec l'entropie croisée
    m = X.shape[0]

    # 1. Gradient de la couche de sortie
    deltay = (Y_pred - Y) / m

    # 2. Gradients des poids et biais de sortie
    gradWy = np.dot(H.T, deltay)
    gradby = np.sum(deltay, axis=0, keepdims=True)

    # 3. Gradient de la couche cachée
    deltah = np.dot(deltay, Wy.T) * H * (1 - H)

    # 4. Gradients des poids et biais cachés
    gradWh = np.dot(X.T, deltah)
    gradbh= np.sum(deltah, axis=0, keepdims=True)
    
    return gradWy, gradby, gradWh, gradbh

<div class="admonition-question admonition">

**Question**

La fonction de coût de l'Eq. (CE) est-elle convexe par rapport aux paramètres $\mathbf{W}$, $\mathbf{b}$ du modèle ? Avec un pas de gradient bien choisi, peut-on assurer la convergence vers le minimum global de la solution ?

</div>

<div class="admonition-question admonition">

**Question**

Compléter le code ci-dessous de sorte à appliquer la descente de gradient sur le perceptron multi-couche défini par les paramètres `Wy, Wh, by, bh`.

</div>

In [14]:
import numpy as np

def accuracy(Wh, bh, Wy,by, images, labels):
    """ W: matrice de paramètres
        b: vecteur de biais
        images: images de MNIST
        labels: étiquettes de MNIST pour les images

        Renvoie l'accuracy du modèle (W, b) sur les images par rapport aux labels
    """
    pred,H = forward(images, Wh, bh, Wy, by)
    return np.where(np.array(pred).argmax(axis=1) != labels.argmax(axis=1), 0.,1.).mean()


N, d = X_train.shape # N exemples, dimension d
hidden_size = 100 # Nombre de neurones de la couche cachée
# Initialisation des poids et des biais
Wh = np.random.randn(d, hidden_size) * np.sqrt(1/d)
Wy = np.random.randn(hidden_size, n_classes) * np.sqrt(1/hidden_size)
by = np.zeros((1, n_classes))
bh = np.zeros((1, hidden_size))

n_epochs = 40 # Nombre d'epochs de la descente de gradient
eta = 0.001 # Learning rate (pas d'apprentissage)
batch_size = 100 # Taille du lot
n_batches = int(float(N) / batch_size)

# Allocation des matrices pour stocker les valeurs des gradients
gradWy = np.zeros((hidden_size, n_classes))
gradWh = np.zeros((d, hidden_size))
gradby = np.zeros((1, n_classes))
gradbh = np.zeros((1, hidden_size))

for epoch in range(n_epochs):
    for batch_idx in range(n_batches):
        # ********* À compléter **********
        # FORWARD PASS : calculer la prédiction y à partir des paramètres courants pour les images du batch
        # BACKWARD PASS :
        # 1) calculer les gradients de l'erreur par rapport à W et b
        # 2) mettre à jour les paramètres W et b selon la descente de gradient
        # Extraction du batch
        batch = X_train[batch_idx*batch_size:(batch_idx+1)*batch_size]
        # Extraction des labels
        labels = Y_train[batch_idx*batch_size:(batch_idx+1)*batch_size]
        
        Y_pred, H = forward(batch, Wh, bh, Wy, by)
        gradWy, gradby, gradWh, gradbh = backward(Y_pred, labels, batch, H, Wy)
        
        loss = cout(labels,Y_pred)
        #print("Loss :", loss)
        
        Wy=Wy-eta*gradWy
        by=by-eta*gradby
        Wh=Wh-eta*gradWh
        bh=bh-eta*gradbh

   
    # Calcul des scores (train, test)
    train_accuracy = accuracy(Wh, bh, Wy,by, X_train, Y_train)
    test_accuracy =  accuracy(Wh, bh, Wy,by, X_test, Y_test)
    # Affichage du score
    print(f"Epoch {epoch}/{n_epochs}: accuracy (train)={train_accuracy*100:.2f}, accuracy (test)={test_accuracy*100:.2f}")


Epoch 0/40: accuracy (train)=30.51, accuracy (test)=31.33
Epoch 1/40: accuracy (train)=45.75, accuracy (test)=46.86
Epoch 2/40: accuracy (train)=51.98, accuracy (test)=53.12
Epoch 3/40: accuracy (train)=56.04, accuracy (test)=57.13
Epoch 4/40: accuracy (train)=58.81, accuracy (test)=59.59
Epoch 5/40: accuracy (train)=60.93, accuracy (test)=61.40
Epoch 6/40: accuracy (train)=62.68, accuracy (test)=63.08
Epoch 7/40: accuracy (train)=63.94, accuracy (test)=64.57
Epoch 8/40: accuracy (train)=65.14, accuracy (test)=65.98
Epoch 9/40: accuracy (train)=66.14, accuracy (test)=67.01
Epoch 10/40: accuracy (train)=67.12, accuracy (test)=67.99
Epoch 11/40: accuracy (train)=67.98, accuracy (test)=68.96
Epoch 12/40: accuracy (train)=68.80, accuracy (test)=69.80
Epoch 13/40: accuracy (train)=69.63, accuracy (test)=70.60
Epoch 14/40: accuracy (train)=70.50, accuracy (test)=71.47
Epoch 15/40: accuracy (train)=71.24, accuracy (test)=72.33
Epoch 16/40: accuracy (train)=72.02, accuracy (test)=73.18
Epoch 1

<div class="admonition-question admonition">

**Question**

Tester en initialisant les matrices à 0 comme pour la régression logistique. Quelles performances obtenez-vous ? Que peut-on en conclure ?

</div>

<div class="admonition-question admonition">

**Question**

Tester deux autres initialisations :

- initialiser les poids avec une loi normale de moyenne nulle et d'écart type à déterminer, par exemple $10^{-1}$,
- initialiser les poids selon la règle de Xavier Glorot [GB10], qui divise la valeur de la gaussienne par $\sqrt{n_i}$, où $n_i$ est le nombre de neurones en entrée de la couche.

</div>

<div class="admonition-question admonition">

**Question**

Évaluer les performances du modèle en classification. Vous devriez obtenir un score d'environ 98% (*accuracy*) sur la base de test pour ce réseau de neurones à une couche cachée.

</div>

[GB10] Xavier Glorot and Yoshua Bengio. Understanding the difficulty of training deep feedforward neural networks. In *In Proceedings of the International Conference on Artificial Intelligence and Statistics (AISTATS'10). Society for Artificial Intelligence and Statistics*. 2010.